# Geospatial Analysis & Site Suitability 

In [ ]:
import pandas as pd

settlements_saws = pd.read_csv("settlements_saws.csv", low_memory = False)

# check if suitability features are present
suitability_features = ["pv_value", "avg_cloud_octas", "avg_max_temp_c", "demand", "population", "distance_to_existing_transmission_lines", "dist_main_road_km", "security_risk_score", "ag_value", "is_underserved"]

print("Feature availability check:")
for feature in suitability_features:
    status = "✅" if feature in settlements_saws.columns else "MISSING"
    print(f"{status} {feature}")

print(f"\nSettlements shape: {settlements_saws.shape}")

## Suitability Score

In [ ]:
from sklearn.preprocessing import MinMaxScaler

df = settlements_saws.copy()

# define features and their direction 
feature_directions = {
    "pv_value": True,
    "avg_cloud_octas": False,
    "avg_max_temp_c": False,
    "demand": True,
    "population": True,
    "dist_main_road_km": False,
    "security_risk_score": False,
    "ag_value": False,
    "is_underserved": True
}

# normalise each feature to 0-1
scaler = MinMaxScaler()

for feature, higher_is_better in feature_directions.items():
    scaled = scaler.fit_transform(df[[feature]])
    if higher_is_better:
        df[f"{feature}_scaled"] = scaled
    else:
        df[f"{feature}_scaled"] = 1-scaled 
        
# verify scaling 
print("Scaled feature ranges (all should be 0.0 to 1.0):")
for feature in feature_directions.keys():
    col = f"{feature}_scaled"
    print(f"{col}: {df[col].min():.2f} - {df[col].max():.2f}")

### Correlation-Based Weighting

In [ ]:
# calculate abs corr of each feature with demand
feature_cols_scaled = [f"{f}_scaled" for f in feature_directions.keys()]

# use demand_scaled as the target 
correlations = df[feature_cols_scaled].corrwith(df["demand_scaled"]).abs()

print("Absolute correlations with demand:")
print(correlations.sort_values(ascending = False).round(4))

# calculate weights 
weights = correlations / correlations.sum()

print("\nDerived weights (sum to 1.0):")
print(weights.sort_values(ascending = False).round(4))
print(f"\nWeight sum: {weights.sum():.4f}")

### Fix Target Leakage 

In [ ]:
# remove demand_scaled from features 
features_for_weighting = [f for f in feature_cols_scaled
                         if f != "demand_scaled"]

# recalculate corrs
correlations_clean = df[features_for_weighting].corrwith(df["demand_scaled"]).abs()

print("Absolute correlations with demand (demand excluded):")
print(correlations_clean.sort_values(ascending = False).round(4))

# recalculate weights 
weights_clean = correlations_clean / correlations_clean.sum()

print("\nDerived weights (sum to 1.0):")
print(weights_clean.sort_values(ascending = False).round(4))
print(f"\nWeight sum: {weights_clean.sum():.4f}")

This didn't fix our weighting problem so we have to think about a weighting solution that balances need and solar potential before calculating correlations

In [ ]:
# composite target 
df["composite_target"] = (
    df["demand_scaled"] * 0.45 + df["pv_value_scaled"] * 0.45 + df["is_underserved_scaled"] * 0.1
)

# correlate all features against composite target
correlations_composite = df[features_for_weighting].corrwith(df["composite_target"]).abs()

print("Absoulte correlations with composite target:")
print(correlations_composite.sort_values(ascending = False).round(4))

# derive weights 
weights_composite = correlations_composite / correlations_composite.sum()

print("\nDerived weights (sum to 1.0):")
print(weights_composite.sort_values(ascending = False).round(4))
print(f"\nWeight sum: {weights_composite.sum():.4f}")

## Suitability Score For All Villages 

In [ ]:
# calculate weighted suitability score 
df["suitability_score"] = (
    df["pv_value_scaled"] * weights_composite["pv_value_scaled"] +
    df["population_scaled"] * weights_composite["population_scaled"] +
    df["avg_cloud_octas_scaled"] * weights_composite["avg_cloud_octas_scaled"] +
    df["security_risk_score_scaled"] * weights_composite["security_risk_score_scaled"] +
    df["avg_max_temp_c_scaled"] * weights_composite["avg_max_temp_c_scaled"] +
    df["ag_value_scaled"] * weights_composite["ag_value_scaled"] +
    df["is_underserved_scaled"] * weights_composite["is_underserved_scaled"] +
    df["dist_main_road_km_scaled"] * weights_composite["dist_main_road_km_scaled"]
)

# verify score range 
print(f"Suitability score range: {df["suitability_score"].min():.4f} - {df["suitability_score"].max():.4f}")
print(f"Mean sitability score: {df["suitability_score"].mean():.4f}")
print(f"Median suitability score: {df["suitability_score"].median():.4f}")

# top 10 suitable villages
top10 = df.nlargest(10, "suitability_score")[["village_name", "province", "district", "suitability_score", "pv_value", "population", "avg_cloud_octas", "security_risk_score", "is_underserved", "lat", "lon"]].reset_index(drop = True)

print("\nTop 10 most suitable villages for solar farm placement:")
print(top10.to_string(index = False))

## 50km Radius around each Top 10 Location

In [ ]:
from math import radians, sin, cos, sqrt, atan2

# haversine calculation function
def haversine(lat1, lon1, lat2, lon2):
     R = 6371
     lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
     dlat = lat2 - lat1
     dlon = lon2 - lon1
     a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
     
     return R * 2 * atan2(sqrt(a), sqrt(1 - a))

# count underserved villages within 50km of top 10
def count_underserved_within_radius(farm_lat, farm_lon, df, radius_km = 50):
    distances = df.apply(lambda row: haversine(farm_lat, farm_lon, row["lat"], row["lon"]), axis = 1)
    within_radius = df[distances <= radius_km]
    
    return pd.Series({
        "villages_within_50km": len(within_radius),
        "underserved_within_50km": within_radius["is_underserved"].sum(),
        "population_served": within_radius["population"].sum(),
        "avg_demand_within_50km": within_radius["demand"].mean().round(1),
        "total_demand_within_50km": within_radius["demand"].sum().round(1)
    })
    
# apply to top 10
print("Calculating transmission coverage for top 10 locations...")
print("(This will take a moment)\n")

coverage = top10.apply(
    lambda row: count_underserved_within_radius(
    row["lat"], row["lon"],df 
    ), 
    axis = 1
)

# attach coverage results 
top10_coverage = pd.concat([top10, coverage], axis = 1)
print("\nTop 10 solar farm locations with transmission coverage:")
print(top10_coverage[[
    "village_name", "province", "suitability_score", "villages_within_50km", "underserved_within_50km", "population_served", "total_demand_within_50km"
]].to_string(index = False))

## Final Ranking Score

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

top10_coverage["underserved_norm"] = scaler.fit_transform(top10_coverage[["underserved_within_50km"]])
top10_coverage["suitability_norm"] = scaler.fit_transform(top10_coverage[["suitability_score"]])
top10_coverage["population_norm"] = scaler.fit_transform(top10_coverage[["population_served"]])
top10_coverage["demand_norm"] = scaler.fit_transform(top10_coverage[["total_demand_within_50km"]])

# final ranking
top10_coverage["final_score"] = (
    top10_coverage["underserved_norm"] * 0.4 +
    top10_coverage["suitability_norm"] * 0.3 +
    top10_coverage["population_norm"] * 0.2 + 
    top10_coverage["demand_norm"] * 0.1
)

# final rank recommendations
final_recommendations = top10_coverage.sort_values("final_score", ascending = False).reset_index(drop = True)
final_recommendations.index += 1 # start ranking from 1

print("FINAL SOLAR FARM LOCATION RECOMMENDATIONS")
print("=" * 151)
print(final_recommendations[["village_name", "province", "suitability_score", "underserved_within_50km", "population_served", "total_demand_within_50km", "final_score"]].to_string())

## Save Final Recommendations

In [ ]:
# not needed anymore 
# final_recommendations.to_csv("solar_farm_recommendations.csv", index = False)
# print("Saved: solar_farm_recommendations.csv")

## SA Map Visualisation for Recommended Farm Locations

In [ ]:
import folium 

# create base map
sa_map_final = folium.Map(
    location = [-28.5, 24.7],
    zoom_start = 6,
    tiles = "CartoDB positron"
)

# colour each farm by province
province_colors = {
    "Northern Cape": "#D85A30",
    "Limpopo": "#1D9E75",
    "Free State": "#378ADD",
    "Gauteng": "#7F77DD",
    "KwaZulu-Natal": "#BA7517"
}

for _, farm in final_recommendations.iterrows():
    village_data = df[(df["village_name"] == farm["village_name"]) & (df["province"] == farm["province"])].iloc[0]
    lat = village_data["lat"]
    lon = village_data["lon"]
    color = province_colors.get(farm["province"], "#888780")
    rank = farm.name # index is the rank
    
    # add 50km transmission radius circle
    folium.Circle(
        location = [lat, lon],
        radius = 500000, # in meters
        color = color,
        fill = True,
        fill_opacity = 0.15,
        weight = 2,
    ).add_to(sa_map_final)
    
    folium.CircleMarker(
        location = [lat, lon],
        radius = 10,
        color = color,
        fill = True,
        fill_opacity = 0.9,
        tooltip = str(farm["village_name"])
    ).add_to(sa_map_final)

# not needed anymore since the error was fixed
# sa_map_final.save("vis12_solar_farm_recommendations.html")
# print("Saved: vis12_solar_farm_recommendations.html")
# print("Open in your browser")

In [ ]:
for _, farm in final_recommendations.iterrows():
    village_data = df[(df["village_name"] == farm["village_name"]) & (df["province"] == farm["province"])].iloc[0]
    print(f"{farm['village_name'][:30]:<30} lat: {village_data['lat']:.4f} lon: {village_data['lon']:.4f}")

### Deduplicate Emthanjeni

In [ ]:
# deduplicate by keeping the highest scoring entry per location 
final_recommendations_deduped = final_recommendations.drop_duplicates(subset = ["village_name", "province"], keep = "first").reset_index(drop = True)
final_recommendations_deduped.index += 1

print(f"Locations after deduplication: {len(final_recommendations_deduped)}")
print(final_recommendations_deduped[["village_name", "province", "final_score"]])

### Add Small Offset to any Remaining Locations

In [ ]:
coords_seen = {}

for idx, farm in final_recommendations_deduped.iterrows():
    village_data = df[(df["village_name"] == farm["village_name"]) & (df["province"] == farm["province"])].iloc[0]
    lat = village_data["lat"]
    lon = village_data["lon"]
    
    # add small offset if coordinates already seen
    key = (round(lat, 3), round(lon, 3))
    if key in coords_seen:
        offset = coords_seen[key] * 0.05
        lat += offset
        lon += offset
        coords_seen[key] += 1
    else:
        coords_seen[key] = 1
        
    final_recommendations_deduped.at[idx, "plot_lat"] = lat
    final_recommendations_deduped.at[idx, "plot_lon"] = lon

print("Coordinates assigned:")
print(final_recommendations_deduped[["village_name", "plot_lat", "plot_lon"]])

### Rebuild the Map

In [ ]:
import folium

# create base map
sa_map_final = folium.Map(
    location = [-28.5, 24.7],
    zoom_start = 6,
    tiles = "CartoDB positron"
)

province_colors = {
    "Northern Cape": "#D85A30",
    "Limpopo": "#1D9E75",
    "Free State": "#378ADD"
}

for _, farm in final_recommendations_deduped.iterrows():
    lat = farm["plot_lat"]
    lon = farm["plot_lon"]
    color = province_colors.get(farm["province"], "#888780")
    rank = farm.name
    
    # 50km radius circle
    folium.Circle(
        location = [lat, lon],
        radius = 50000, 
        color = color,
        fill = True,
        fill_opacity = 0.15,
        weight = 2
    ).add_to(sa_map_final)
    
    # Marker with full tooltip 
    folium.CircleMarker(
        location = [lat, lon],
        radius = 10,
        color = color,
        fill = True,
        fill_opacity = 0.9,
        tooltip = (
            f"Rank #{rank} | "
            f"{farm['village_name']} | "
            f"{farm['province']} | "
            f"Final Score: {farm['final_score']:.3f} | "
            f"Underserved within 50km: {int(farm['underserved_within_50km'])} | "
            f"Population served: {int(farm['population_served'])}"
        )
    ).add_to(sa_map_final)
    
# legend
legend_html = """
<div style="position:fixed;bottom:30px;left:30px;z-index:1000;
     background:white;padding:12px;border-radius:8px;
     border:1px solid #ccc;font-size:12px;">
  <b>Solar Farm Recommendations</b><br><br>
  <span style="color:#D85A30;">●</span> Northern Cape<br>
  <span style="color:#1D9E75;">●</span> Limpopo<br>
  <span style="color:#378ADD;">●</span> Free State<br>
  <br>
  <span style="opacity:0.4;">○</span> 50km transmission radius
</div>
"""

sa_map_final.get_root().html.add_child(folium.Element(legend_html))
sa_map_final.save("vis12_solar_farm_recommendations.html")
print("Saved: vis12_solar_recommendations.html")
    

## Save Geospatial Analysis & Site Suitability

In [ ]:
final_recommendations_deduped.to_csv("solar_farm_recommendations_final.csv", index = False)
df.to_csv("settlements_saws_scored.csv", index = False)

print("Phase 4 outputs saved:")
print("solar_farm_recommendations_final.csv")
print("settlements_saws_score.csv")

print("\nPhase 4 Summary:")
print(f"Villages score: {len(df)}")
print(f"Suitability score range: {df['suitability_score'].min():.4f} - {df['suitability_score'].max():.4f}")
print(f"Recommended locations: {len(final_recommendations_deduped)}")
print(f"Provinces covered: {final_recommendations_deduped['province'].unique().tolist()}")
print(f"Total population served: {final_recommendations_deduped['population_served'].sum()}")
print(f"Total underserved villages within coverage: {final_recommendations_deduped['underserved_within_50km'].sum()}")